# TP — Traitement Distribué avec Apache Spark

---

Un diamantaire reçoit chaque jour des dizaines de milliers de transactions :
- *Quel type de taille (cut) génère le plus de revenus ?*
- *Quelle combinaison couleur/clarté atteint les meilleurs prix moyens ?*
- *Les diamants "Ideal" sont-ils vraiment les plus chers ?*

**Apache Spark** distribue ces calculs sur un cluster et répond en millisecondes, même sur des millions de lignes.

**Objectifs de ce TP** :
- Comprendre la relation entre les 3 APIs Spark : **RDD**, **DataFrame**, **Spark SQL**
- Écrire des pipelines de traitement distribués
- Savoir quand utiliser l'API RDD, l'API DataFrame ou Spark SQL

> Ce TP tourne sur **Google Colab** en mode local — les concepts s'appliquent identiquement sur un vrai cluster.

**Dataset** : `diamonds` (sklearn / OpenML) — 53 940 diamants avec 10 caractéristiques.

---
## Partie 1 — Théorie

---

### 1.1 — Architecture Spark

Spark fonctionne en mode **maître / travailleurs** :

```
┌─────────────────────────────────────────────────────┐
│                     Driver                          │
│  SparkSession  ──►  DAG Scheduler  ──►  Task Sched │
└──────────────────────────┬──────────────────────────┘
                           │ distribue les tâches
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
     Executor 1       Executor 2       Executor 3
     [Partition 1]    [Partition 2]    [Partition 3]
     [Partition 4]    [Partition 5]    [Partition 6]
```

| Composant | Rôle |
|---|---|
| **Driver** | Contient ton code, construit le plan d'exécution (DAG) |
| **SparkSession** | Point d'entrée unique depuis Spark 2.0 |
| **Executor** | Processus JVM sur chaque nœud — exécute les tâches |
| **Partition** | Fragment de données traité par 1 executor à la fois |

> **Règle clé** : 1 partition = 1 tâche = 1 executor au moment T

### 1.2 — RDD : le fondement de Spark

**RDD** (*Resilient Distributed Dataset*) est l'abstraction de base : une collection d'objets **immuable**, **partitionnée** sur le cluster, **recalculable** en cas de panne.

#### Deux types d'opérations

| Type | Caractéristique | Exemples |
|---|---|---|
| **Transformations** | *Lazy* — construisent le DAG sans calculer | `map`, `filter`, `flatMap`, `reduceByKey` |
| **Actions** | *Eager* — déclenchent le calcul réel | `collect`, `count`, `take`, `saveAsTextFile` |

#### Lazy Evaluation

```python
rdd = sc.textFile("data.csv")            # rien ne se passe
rdd2 = rdd.map(lambda l: l.split(","))   # DAG s'enrichit
rdd3 = rdd2.filter(lambda r: r[6] > 500) # DAG s'enrichit
rdd3.count()                             # ← ICI Spark calcule tout
```

> Avantage : Spark optimise **l'ensemble du pipeline** avant d'exécuter quoi que ce soit.

### 1.3 — DataFrame : RDD avec un schéma

Depuis Spark 2.0, le **DataFrame** est l'API recommandée.

```
RDD[Any]     →  pas de schéma, pas d'optimisation automatique
Dataset[T]   →  typé, Scala/Java uniquement
DataFrame    →  Dataset[Row] : colonnes nommées et typées
                API Python + optimiseur Catalyst intégré
```

**Ce que Catalyst fait pour toi** :
- Pousse les filtres le plus tôt possible pour réduire les données
- Fusionne les étapes compatibles en un seul passage
- Choisit automatiquement le meilleur type de jointure

```python
# RDD : tu décris COMMENT faire
rdd.filter(lambda r: float(r[6]) > 1000).map(lambda r: (r[1], float(r[6])))

# DataFrame : tu décris QUOI faire — Catalyst décide COMMENT
df.filter("price > 1000").select("cut", "price")
```

### 1.4 — Comparaison des 3 APIs

| Critère | RDD | DataFrame | Spark SQL |
|---|---|---|---|
| **Niveau** | Bas niveau | Haut niveau | Haut niveau |
| **Schéma** | Aucun | Colonnes typées | Colonnes typées |
| **Optimisation** | Manuelle | Catalyst auto | Catalyst auto |
| **Style** | Python fonctionnel | API chainée | SQL textuel |
| **Cas d'usage** | Texte brut, logique complexe | Données tabulaires | Requêtes lisibles |

> **Règle pratique** : **DataFrame par défaut**. Passe en RDD si l'API DataFrame ne suffit pas. Spark SQL pour les requêtes complexes ou les profils SQL.

### 1.5 — Pipeline de traitement

```
Source (CSV / Parquet / BDD / API)
         ↓  spark.read  ou  sc.textFile
    RDD / DataFrame
         ↓  filter / withColumn / groupBy / join
    DataFrame transformé
         ↓  createOrReplaceTempView
    Vue SQL  ──►  spark.sql("SELECT ...")
         ↓  write / show / collect
    Résultat final
```

DataFrame et Spark SQL sont **interchangeables** — Catalyst les optimise de la même façon.

---
## Partie 2 — Pratique

---

### Étape 0 — Installation et configuration

In [ ]:
!pip install pyspark --quiet
print("PySpark installé.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import time
import pandas as pd
import numpy as np

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField,
                                IntegerType, StringType, DoubleType)
from pyspark.sql.window import Window
from pyspark import StorageLevel
print("Imports OK.")

---
### Étape 1 — SparkSession et SparkContext

**SparkSession** est le point d'entrée unique depuis Spark 2.0.
Il expose à la fois l'API DataFrame/SQL et le SparkContext (API RDD).

In [ ]:
spark = (SparkSession.builder
         .appName("TP-Spark-Diamonds")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"Spark version : {spark.version}")
print(f"Maître        : {sc.master}")
print(f"Cœurs dispo   : {sc.defaultParallelism}")

---
### Étape 2 — Chargement du dataset Diamonds

Le dataset **Diamonds** (sklearn / OpenML) contient 53 940 diamants décrits par 10 caractéristiques.

| Colonne | Type | Description |
|---|---|---|
| `carat` | float | Poids du diamant |
| `cut` | str | Qualité de la taille : Fair, Good, Very Good, Premium, Ideal |
| `color` | str | Couleur : D (meilleure) → J (moins bonne) |
| `clarity` | str | Clarté : IF (parfaite) → I1 (inclusions visibles) |
| `depth` | float | Profondeur totale en % |
| `table` | float | Largeur du dessus en % |
| `price` | int | Prix en dollars |
| `x`, `y`, `z` | float | Dimensions en mm |

In [ ]:
from sklearn.datasets import fetch_openml

print("Téléchargement du dataset diamonds via OpenML...")
diamonds = fetch_openml(name='diamonds', version=1, as_frame=True)
df_pd = diamonds.frame

# Convertir les colonnes catégorielles en string
for col in ['cut', 'color', 'clarity']:
    df_pd[col] = df_pd[col].astype(str)

df_pd['price'] = df_pd['price'].astype(int)

# Sauvegarder en CSV pour le chargement Spark
df_pd.to_csv("diamonds.csv", index=False)

print(f"Dataset chargé : {df_pd.shape[0]:,} lignes × {df_pd.shape[1]} colonnes")
df_pd.head(3)

In [ ]:
# Charger en Spark DataFrame avec schéma explicite
schema = StructType([
    StructField("carat",   DoubleType(),  True),
    StructField("cut",     StringType(),  True),
    StructField("color",   StringType(),  True),
    StructField("clarity", StringType(),  True),
    StructField("depth",   DoubleType(),  True),
    StructField("table",   DoubleType(),  True),
    StructField("price",   IntegerType(), True),
    StructField("x",       DoubleType(),  True),
    StructField("y",       DoubleType(),  True),
    StructField("z",       DoubleType(),  True),
])

df = spark.read.schema(schema).option("header", True).csv("diamonds.csv")

print(f"Spark DataFrame : {df.count():,} lignes")
print(f"Partitions      : {df.rdd.getNumPartitions()}")
df.printSchema()

In [ ]:
# Aperçu des données
df.show(5)

In [ ]:
# Distribution par qualité de taille
df.groupBy("cut").count().orderBy("count", ascending=False).show()

---
### Étape 3 — API RDD

On travaille au bas niveau pour comprendre ce qui se passe sous le capot.

#### 3.1 — Créer un RDD

In [ ]:
# Depuis le CSV (chaque ligne = une chaîne de caractères)
rdd_raw = sc.textFile("diamonds.csv")

print("Type       :", type(rdd_raw))
print("Partitions :", rdd_raw.getNumPartitions())
print("Lignes     :", rdd_raw.count())
print()
print("Premières lignes brutes :")
for l in rdd_raw.take(3):
    print(" ", l)

In [ ]:
# Parser le CSV manuellement — retirer l'en-tête
header = rdd_raw.first()
rdd = (rdd_raw
    .filter(lambda l: l != header)        # enlever l'en-tête
    .map(lambda l: l.split(","))          # splitter en liste
)
print("Nombre de diamants :", rdd.count())
print("Premier enregistrement :", rdd.first())

#### 3.2 — Transformations : `map`, `filter`, `flatMap`

In [ ]:
# map : extraire 2 colonnes (cut=index 1, price=index 6)
rdd_cut_price = rdd.map(lambda r: (r[1], int(r[6])))
print("Extrait (cut, price) :")
print(rdd_cut_price.take(5))

In [ ]:
# filter : garder uniquement les diamants Ideal
rdd_ideal = rdd.filter(lambda r: r[1] == "Ideal")
print(f"Diamants Ideal : {rdd_ideal.count():,}")

# filter + map chainés
rdd_ideal_prix = rdd_ideal.map(lambda r: int(r[6]))
print(f"Prix moyen Ideal : ${sum(rdd_ideal_prix.collect()) / rdd_ideal.count():,.0f}")

In [ ]:
# flatMap : extraire toutes les valeurs catégorielles (cut, color, clarity)
# pour observer les modalités présentes dans le dataset
categories = (rdd
    .flatMap(lambda r: [r[1], r[2], r[3]])   # cut, color, clarity
    .distinct()
    .collect()
)
print(f"Modalités distinctes ({len(categories)}) : {sorted(categories)}") 

#### 3.3 — Agrégations : `reduceByKey`

In [ ]:
# Compter les diamants par qualité de taille
count_by_cut = (rdd
    .map(lambda r: (r[1], 1))                # (cut, 1)
    .reduceByKey(lambda a, b: a + b)         # additionner
    .sortBy(lambda x: x[1], ascending=False)
)
print("Nombre de diamants par cut :")
for cut, n in count_by_cut.collect():
    print(f"  {cut:<12} : {n:,}")

In [ ]:
# Prix total et nombre de ventes par cut
# (cut, prix) → reduceByKey pour sommer les prix et compter
rdd_cut_stats = (rdd
    .map(lambda r: (r[1], (int(r[6]), 1)))          # (cut, (prix, 1))
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) # sommer prix et compteur
    .map(lambda x: (x[0], x[1][0], x[1][1], round(x[1][0]/x[1][1], 2)))
    .sortBy(lambda x: x[3], ascending=False)         # trier par prix moyen
)

print(f"{'Cut':<12} {'CA total':>12} {'Nb':>7} {'Prix moyen':>12}")
print("-" * 47)
for cut, total, nb, moy in rdd_cut_stats.collect():
    print(f"{cut:<12} ${total:>11,.0f} {nb:>7,} ${moy:>11,.2f}")

#### 3.4 — Lazy Evaluation : observation directe

In [ ]:
# Les transformations ne calculent rien — elles construisent le DAG
rdd_pipeline = (rdd
    .filter(lambda r: r[2] == "D")           # couleur D (meilleure)
    .map(lambda r: int(r[6]))               # prix
    .filter(lambda p: p > 5000)             # prix > 5000$
)
print("Après 3 transformations : rien n'a encore été calculé.")
print("Type :", type(rdd_pipeline))

# L'action déclenche TOUT le calcul
t0 = time.time()
n = rdd_pipeline.count()
print(f"
Après count() : {n:,} diamants couleur D avec prix > 5 000$")
print(f"Calculé en {time.time()-t0:.3f}s")

---
### Étape 4 — API DataFrame

Même objectif qu'à l'étape 3, avec l'API haut niveau. Catalyst optimise automatiquement.

#### 4.1 — Exploration

In [ ]:
# Statistiques descriptives
df.select("carat", "depth", "price").describe().show()

In [ ]:
# Distribution des prix par catégorie de taille
df.groupBy("cut").agg(
    F.count("*").alias("nb"),
    F.round(F.min("price"),  2).alias("prix_min"),
    F.round(F.avg("price"),  2).alias("prix_moyen"),
    F.round(F.max("price"),  2).alias("prix_max"),
).orderBy(F.col("prix_moyen").desc()).show()

#### 4.2 — Sélection et filtrage

In [ ]:
# select + filter combinés
df.select("cut", "color", "clarity", "carat", "price")   .filter((F.col("price") > 5000) & (F.col("color") == "D"))   .show(5)

In [ ]:
# Compter par catégorie avec filtre
print("Diamants couleur D avec prix > 5 000$ :")
df.filter((F.col("color") == "D") & (F.col("price") > 5000))   .groupBy("cut")   .count()   .orderBy("count", ascending=False)   .show()

#### 4.3 — Nouvelles colonnes avec `withColumn`

In [ ]:
# Catégorie de prix
df = df.withColumn("gamme",
    F.when(F.col("price") <  1000, "Entrée de gamme")
     .when(F.col("price") <  5000, "Milieu de gamme")
     .when(F.col("price") < 15000, "Haut de gamme")
     .otherwise("Luxe")
)

# Volume approximatif (x * y * z)
df = df.withColumn("volume", F.round(F.col("x") * F.col("y") * F.col("z"), 3))

df.select("cut", "carat", "price", "gamme", "volume").show(6)

#### 4.4 — Agrégations

In [ ]:
# Prix moyen par cut et color
df.groupBy("cut", "color").agg(
    F.round(F.avg("price"), 2).alias("prix_moyen"),
    F.count("*").alias("nb")
).orderBy("cut", "color").show(10)

In [ ]:
# Pivot : prix moyen par color (lignes) et cut (colonnes)
df.groupBy("color")   .pivot("cut", ["Fair", "Good", "Very Good", "Premium", "Ideal"])   .agg(F.round(F.avg("price"), 0))   .orderBy("color")   .show()

In [ ]:
# Gamme de prix la plus représentée par cut
df.groupBy("cut", "gamme").count()   .orderBy("cut", F.col("count").desc())   .show(12)

#### 4.5 — Jointures

In [ ]:
# Table de référence : rang de qualité par cut
cut_ref = spark.createDataFrame([
    ("Fair",      1, "Taille basique — proportions non optimales"),
    ("Good",      2, "Bonne taille — réflexion acceptable"),
    ("Very Good", 3, "Très bonne taille — excellent rapport qualité/prix"),
    ("Premium",   4, "Taille premium — proche de l'Ideal"),
    ("Ideal",     5, "Taille idéale — réflexion maximale de la lumière"),
], ["cut", "rang_qualite", "description"])

cut_ref.show(truncate=False)

In [ ]:
# Jointure diamonds × cut_ref
df_enrichi = df.join(cut_ref, on="cut", how="left")

df_enrichi.select("cut", "rang_qualite", "color", "carat", "price")            .orderBy("rang_qualite", F.col("price").desc())            .show(8)

In [ ]:
# Prix moyen par rang de qualité
df_enrichi.groupBy("rang_qualite", "cut").agg(
    F.round(F.avg("price"), 2).alias("prix_moyen"),
    F.count("*").alias("nb")
).orderBy("rang_qualite").show()

---
### Étape 5 — Spark SQL

Spark SQL permet d'interroger un DataFrame avec du **SQL standard**.
Le résultat est toujours un DataFrame — les deux APIs sont interchangeables.

In [ ]:
# Enregistrer les vues temporaires
df_enrichi.createOrReplaceTempView("diamonds")
cut_ref.createOrReplaceTempView("cut_ref")
print("Vues créées : diamonds, cut_ref")

In [ ]:
# Requête 1 : prix moyen et médian par cut
spark.sql('''
    SELECT cut,
           COUNT(*)                      AS nb,
           ROUND(AVG(price), 2)          AS prix_moyen,
           ROUND(PERCENTILE(price, 0.5)) AS prix_median,
           ROUND(MAX(price), 2)          AS prix_max
    FROM diamonds
    GROUP BY cut
    ORDER BY prix_moyen DESC
''').show()

In [ ]:
# Requête 2 : top 5 combinaisons cut/color par prix moyen
spark.sql('''
    SELECT cut, color,
           COUNT(*)             AS nb,
           ROUND(AVG(price), 2) AS prix_moyen,
           ROUND(AVG(carat), 3) AS carat_moyen
    FROM diamonds
    WHERE price > 1000
    GROUP BY cut, color
    HAVING COUNT(*) > 50
    ORDER BY prix_moyen DESC
    LIMIT 10
''').show()

In [ ]:
# Requête 3 : comparaison cut vs rang de qualité (jointure SQL)
spark.sql('''
    SELECT d.cut,
           r.rang_qualite,
           COUNT(*)             AS nb_diamants,
           ROUND(AVG(d.price), 2) AS prix_moyen,
           ROUND(AVG(d.carat), 3)  AS carat_moyen
    FROM diamonds d
    JOIN cut_ref r ON d.cut = r.cut
    GROUP BY d.cut, r.rang_qualite
    ORDER BY r.rang_qualite
''').show()

In [ ]:
# Requête 4 : diamants au-dessus du prix moyen de leur catégorie (sous-requête)
spark.sql('''
    SELECT cut, color, carat, price, prix_moyen_cut,
           ROUND(price - prix_moyen_cut, 2) AS ecart
    FROM (
        SELECT *,
               ROUND(AVG(price) OVER (PARTITION BY cut), 2) AS prix_moyen_cut
        FROM diamonds
    )
    WHERE price > prix_moyen_cut
    ORDER BY ecart DESC
    LIMIT 10
''').show()

---
### Étape 6 — Passage RDD ↔ DataFrame

Les deux APIs sont pleinement interopérables.

In [ ]:
# DataFrame → RDD (chaque élément devient une Row)
rdd_from_df = df.select("cut", "color", "price").rdd
print("Type :", type(rdd_from_df))
print("Premiers éléments :")
for row in rdd_from_df.take(3):
    print(f"  cut={row['cut']}, color={row['color']}, price={row['price']}$")

In [ ]:
# Traitement RDD : prix max par cut
top_prix = (rdd_from_df
    .map(lambda r: (r["cut"], r["price"]))
    .reduceByKey(lambda a, b: max(a, b))
    .sortBy(lambda x: x[1], ascending=False)
)
print("Prix maximum par cut (via RDD) :")
for cut, pmax in top_prix.collect():
    print(f"  {cut:<12} : ${pmax:,}")

In [ ]:
# RDD → DataFrame avec schéma
rdd_stats = sc.parallelize([
    ("Fair",      "Basique",    1),
    ("Good",      "Standard",   2),
    ("Very Good", "Premium+",   3),
    ("Premium",   "Haut gamme", 4),
    ("Ideal",     "Excellence", 5),
])

schema = StructType([
    StructField("cut",      StringType(),  True),
    StructField("label",    StringType(),  True),
    StructField("rang",     IntegerType(), True),
])

df_from_rdd = spark.createDataFrame(rdd_stats, schema)
df_from_rdd.show()

---
### Étape 7 — Cache et Persistance

Quand un DataFrame est **réutilisé plusieurs fois**, le mettre en cache évite de recalculer le pipeline depuis le début.

#### 7.1 — Sans cache : le pipeline est recalculé à chaque action

In [ ]:
# Pipeline avec jointure (un peu coûteux)
df_analyse = (df
    .filter(F.col("price") > 2000)
    .join(cut_ref, on="cut")
    .withColumn("prix_carat", F.round(F.col("price") / F.col("carat"), 2))
    .groupBy("cut", "rang_qualite")
    .agg(F.round(F.avg("prix_carat"), 2).alias("prix_par_carat"),
         F.count("*").alias("nb"))
)

t0 = time.time(); n1 = df_analyse.count(); t1 = time.time()
print(f"Sans cache — 1er appel  : {t1-t0:.3f}s")

t0 = time.time(); n2 = df_analyse.count(); t1 = time.time()
print(f"Sans cache — 2ème appel : {t1-t0:.3f}s  (recalcul complet)")

#### 7.2 — Avec cache : le résultat est réutilisé

In [ ]:
df_cache = (df
    .filter(F.col("price") > 2000)
    .join(cut_ref, on="cut")
    .withColumn("prix_carat", F.round(F.col("price") / F.col("carat"), 2))
    .groupBy("cut", "rang_qualite")
    .agg(F.round(F.avg("prix_carat"), 2).alias("prix_par_carat"),
         F.count("*").alias("nb"))
    .cache()
)

t0 = time.time(); n1 = df_cache.count(); t1 = time.time()
print(f"Avec cache — 1er appel  : {t1-t0:.3f}s  (calcul + mise en cache)")

t0 = time.time(); n2 = df_cache.count(); t1 = time.time()
print(f"Avec cache — 2ème appel : {t1-t0:.3f}s  (lecture depuis le cache) ✓")

df_cache.orderBy("rang_qualite").show()
df_cache.unpersist()

#### 7.3 — Niveaux de persistance

| Niveau | RAM | Disque | Utilisation |
|---|---|---|---|
| `MEMORY_ONLY` | ✓ | ✗ | Données tiennent en RAM |
| `MEMORY_AND_DISK` | ✓ | ✓ (si RAM insuffisante) | **Recommandé** |
| `DISK_ONLY` | ✗ | ✓ | RAM très limitée |
| `MEMORY_ONLY_2` | ✓ (×2) | ✗ | Réplication pour la tolérance aux pannes |

In [ ]:
df_persist = df.filter(F.col("cut") == "Ideal").persist(
    StorageLevel.MEMORY_AND_DISK
)
df_persist.count()   # déclenche le calcul + persistance
print(f"Diamants Ideal persistés : {df_persist.count():,}")
df_persist.unpersist()
print("Mémoire libérée.")

---
### Bilan

| API | Style | Quand l'utiliser |
|---|---|---|
| **RDD** | `rdd.map(...)` | Texte brut, logique complexe ligne par ligne |
| **DataFrame** | `df.groupBy(...)` | Données tabulaires — usage principal |
| **Spark SQL** | `spark.sql("SELECT...")` | Requêtes complexes, profils SQL |
| **Cache** | `.cache()` | DataFrame réutilisé plusieurs fois |

#### Règles à retenir

1. **DataFrame par défaut** — passe en RDD uniquement si nécessaire
2. **`reduceByKey` plutôt que `groupByKey`** — réduit avant le shuffle
3. **Définir le schéma explicitement** — `inferSchema=True` est lent en production
4. **Cache si réutilisation** — évite de recalculer le pipeline
5. **DataFrame et Spark SQL sont équivalents** — Catalyst les traite pareil

In [ ]:
spark.stop()
print("Session Spark fermée.")